# 15. Lining up other datasets

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

Many hand datasets also publish 3D joint coordinates. The count, the order, and the definition of a fingertip differ from MANO's 16 joints. A Euclidean error means something only after both sides name the same anatomical points.

The three datasets below show up often in hand-pose benchmarks. The benchmark itself reads points that started as pixels plus depth and were then moved into the camera frame, in millimetres. It does not read MANO's 48-D pose.

## Three annotations

| Data | Points used for the score | How they differ from MANO |
|---|---|---|
| ICVL | 16 | Each finger has root, middle, and tip, plus a palm point. There is no DIP slot |
| NYU | 14 of 36 points | The indices used are 0, 3, 6, 9, 12, 15, 18, 21, 24, 25, 27, 30, 31, 32 |
| MSRA | 21 | Wrist, then index, middle, ring, pinky, thumb. Each finger has four points: MCP, PIP, DIP, tip |

MSRA's 21 points are the easiest to confuse with "MANO's 16 joints plus 5 fingertips", because the counts match. Three differences remain:

1. Every MSRA finger includes the tip. MANO joints 0 through 15 do not. The tips are vertices 744, 320, 443, 554, and 671.
2. MSRA's finger order is index, middle, ring, pinky, thumb. MANO stores the pinky before the ring, after the middle finger.
3. What "wrist" means, and which mesh point is the tip, follows each dataset's own rule. A vertex id means something only on the MANO mesh.

The next cell lays MANO's naming beside MSRA's order. The first four positions agree. They split apart at the index tip.


In [ ]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

from official_mano import require_model

model = require_model()
flat = model.forward(flat_hand=True)

mano_names = [
    "wrist",
    "index MCP", "index PIP", "index DIP",
    "middle MCP", "middle PIP", "middle DIP",
    "pinky MCP", "pinky PIP", "pinky DIP",
    "ring MCP", "ring PIP", "ring DIP",
    "thumb CMC", "thumb MCP", "thumb IP",
]
msra_order = []
for finger in ("index", "middle", "ring", "pinky", "thumb"):
    msra_order.extend([f"{finger} MCP", f"{finger} PIP", f"{finger} DIP", f"{finger} tip"])
msra_names = ["wrist"] + msra_order

print(f"{'index':<8}{'MANO joints 0-15':<24}{'MSRA'}")
for index in range(21):
    left = mano_names[index] if index < len(mano_names) else "(no MANO joint)"
    print(f"{index:<8}{left:<24}{msra_names[index]}")

same = [i for i in range(16) if mano_names[i] == msra_names[i]]
print("indices whose anatomical names still match:", same)


index   MANO joints 0-15        MSRA
0       wrist                   wrist
1       index MCP               index MCP
2       index PIP               index PIP
3       index DIP               index DIP
4       middle MCP              index tip
5       middle PIP              middle MCP
6       middle DIP              middle PIP
7       pinky MCP               middle DIP
8       pinky PIP               middle tip
9       pinky DIP               ring MCP
10      ring MCP                ring PIP
11      ring PIP                ring DIP
12      ring DIP                ring tip
13      thumb CMC               pinky MCP
14      thumb MCP               pinky PIP
15      thumb IP                pinky DIP
16      (no MANO joint)         pinky tip
17      (no MANO joint)         thumb MCP
18      (no MANO joint)         thumb PIP
19      (no MANO joint)         thumb DIP
20      (no MANO joint)         thumb tip
indices whose anatomical names still match: [0, 1, 2, 3]


## The order of an alignment

1. Read MANO's 16 joints and 5 fingertips.
2. Permute them into the other side's order. For MSRA, the tips are those 5 vertices, and the pinky and ring swap so the ring comes before the pinky.
3. Move that list into the other camera frame, and convert metres to millimetres.
4. Then compute the Euclidean distance, point by point.

Subtracting indices 0 through 15 directly skips steps 2 and 3. The subtraction compares different anatomical sites, and the error does not mean a pose error.

The next cell actually stacks 21 points in MSRA order. The unit is millimetres. The origin is still MANO's model frame. Nothing has been moved into a camera yet.


In [ ]:
joints = flat["joints"]
tips = flat["tips"]
# MANO's first thumb joint is the CMC. It is placed in MSRA's thumb-MCP slot here only to show the order.
# A real alignment still has to check how that dataset defines the base of the thumb.
msra_points = [
    joints[0],
    joints[1], joints[2], joints[3], tips["index"],
    joints[4], joints[5], joints[6], tips["middle"],
    joints[10], joints[11], joints[12], tips["ring"],
    joints[7], joints[8], joints[9], tips["pinky"],
    joints[13], joints[14], joints[15], tips["thumb"],
]
msra_mm = np.stack(msra_points) * 1000
print("21 points in MSRA order, millimetres, shape", msra_mm.shape)
print("index 4 is the index tip", np.round(msra_mm[4], 1))
print("MANO joint 4 is the middle MCP", np.round(joints[4] * 1000, 1))
print("distance between those two points, mm", round(float(np.linalg.norm(msra_mm[4] - joints[4] * 1000)), 1))


21 points in MSRA order, millimetres, shape (21, 3)
index 4 is the index tip [-72.7   5.1  28.5]
MANO joint 4 is the middle MCP [1.  4.9 2.8]
distance between those two points, mm 78.1


MSRA indices 0 through 3 are the wrist and the three index joints. They are the same anatomical sites as MANO joints 0 through 3. Index 4 is the index tip. MANO joint 4 is already the middle MCP. The distance printed above is that mismatch.

The next chapter collects the tutorial into questions. Cover the answers and write them yourself.
